## MMTHE01 - Masters Thesis

### D. Thesis - Create a working model - with SMOTE
* Applying SMOTE to the data
* When applying SMOTE, it is best not to use one-hot-encoding for categorical data. Therefore the data is imported before encoding and and all categorical 

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import time
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder

In [2]:
### Import libraries to save models
import pickle
from tensorflow.keras.models import Model, load_model

In [3]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Load Dataset

In [4]:
with open("saved_data/Oversampled_datasets_folder.pkl","rb") as f:
    X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled = pickle.load(f)

#### Loading the models

In [5]:
# load Random Forest model
with open("saved_models/random_forest_model_Oversampled.pkl", "rb") as f:
    rf = pickle.load(f)

In [6]:
# load XGBoost model
xgb = xgb.XGBClassifier()
xgb.load_model("saved_models/xgb_model_Oversampled.json")

In [7]:
# load ANN model
model = load_model("saved_models/ann_model_Oversampled.keras")

In [8]:
# load SVM model
with open("saved_models/svm_pipeline_Oversampled.pkl", "rb") as f:
    pipeline = pickle.load(f)

In [9]:
# load GBM model
with open("saved_models/gbm_model_Oversampled.pkl", "rb") as f:
    gbm = pickle.load(f)

### 4.6 Cross Validation Checks

In [10]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, f1_score

In [11]:
# Define multiple scoring metrics
scoring = {
    'auc': 'roc_auc',
    'recall': 'recall',
    'precision': 'precision',
    'F1': make_scorer(f1_score, average='macro')
}

#### 4.6.1 Cross Validation for the Random Forest Model

In [12]:
# Run cross-validation once
cv_results_rf = cross_validate(
    estimator=rf,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [13]:
# Print results - RF AUC
print("AUC Random Forest: {:.3f} %".format(cv_results_rf['test_auc'].mean() * 100))
print("AUC Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_auc'].std()))

AUC Random Forest: 88.846 %
AUC Standard Deviation Random Forest: 0.10380


In [14]:
# Print results - RF Recall
print("Recall Random Forest: {:.4f}".format(cv_results_rf['test_recall'].mean()))
print("Recall Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_recall'].std()))

Recall Random Forest: 0.7459
Recall Standard Deviation Random Forest: 0.26949


In [15]:
# Print results - RF precision
print("Precision Random Forest: {:.4f}".format(cv_results_rf['test_precision'].mean()))
print("Precision Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_precision'].std()))

Precision Random Forest: 0.6516
Precision Standard Deviation Random Forest: 0.08381


In [16]:
# Print results - Random Forest F1 Score
print("F1-score Random Forest: {:.4f}".format(cv_results_rf['test_F1'].mean()))
print("F1-score Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_F1'].std()))

F1-score Random Forest: 0.7715
F1-score Standard Deviation Random Forest: 0.11055


#### 4.6.2 Cross Validation for the XGBoost Model

In [17]:
# Run cross-validation once
cv_results_xgb = cross_validate(
    estimator=xgb,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [18]:
# Print results - XGB AUC
print("AUC XGBoost: {:.3f} %".format(cv_results_xgb['test_auc'].mean() * 100))
print("AUC Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_auc'].std()))

AUC XGBoost: 92.859 %
AUC Standard Deviation XGBoost: 0.10445


In [19]:
# Print results - XGB Recall
print("Recall XGBoost: {:.4f}".format(cv_results_xgb['test_recall'].mean()))
print("Recall Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_recall'].std()))

Recall XGBoost: 0.7011
Recall Standard Deviation XGBoost: 0.41739


In [20]:
# Print results - XGB Precision
print("Precision XGBoost: {:.4f}".format(cv_results_xgb['test_precision'].mean()))
print("Precision Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_precision'].std()))

Precision XGBoost: 0.8877
Precision Standard Deviation XGBoost: 0.14079


In [21]:
# Print results - XGB F1 Score
print("F1-score XGBoost: {:.4f}".format(cv_results_xgb['test_F1'].mean()))
print("F1-score Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_F1'].std()))

F1-score XGBoost: 0.8257
F1-score Standard Deviation XGBoost: 0.23503


#### 4.6.3 Cross Validation for the ANN Model

In [22]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold

# --------------------------
# 1. Define your ANN model builder
# --------------------------
def create_ann_model(input_dim):
    # Build an ANN model
    model = Sequential([
        Input(shape=(X_train.shape[1],)),  # Explicit Input layer instead of input_dim in Dense
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    
    # Early stopping to prevent overfitting
    early_stop = EarlyStopping(monitor='val_AUC', patience=3, restore_best_weights=True, mode='max')

    return model

In [23]:
# --------------------------
# 2. Manual cross-validation loop
# --------------------------
def cross_validate_ann(X, y):

    kf = KFold(n_splits=10, shuffle=True, random_state=1)

    auc_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]


        classes = np.unique(y_train)
        class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
        class_weights = {i: class_weights[i] for i in range(len(classes))}

        # Build model fresh each fold
        model = create_ann_model(input_dim=X.shape[1])

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

        # Early stopping to prevent overfitting
        early_stop = EarlyStopping(monitor='val_AUC', patience=3, restore_best_weights=True, mode='max')

        # Train
            # Train the ANN model with timing
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=50,
            batch_size=256,
            callbacks=[early_stop],
            class_weight=class_weights,
            verbose=1
        )
    

        # Predict probabilities for AUC
        y_prob = model.predict(X_test).ravel()

        # Convert to class labels
        y_pred = (y_prob >= 0.41000000000000003).astype(int)

        # Compute metrics
        auc_scores.append(roc_auc_score(y_test, y_prob))
        recall_scores.append(recall_score(y_test, y_pred))
        precision_scores.append(precision_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred, average='macro'))

    # Return results as a dictionary similar to sklearn
    return {
        "test_auc": np.array(auc_scores),
        "test_recall": np.array(recall_scores),
        "test_precision": np.array(precision_scores),
        "test_F1": np.array(f1_scores)
    }

In [24]:
cv_results_ann = cross_validate_ann(X_train_scaled, y_train)

Epoch 1/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - AUC: 0.6464 - loss: 0.4822 - val_AUC: 0.0000e+00 - val_loss: 1.0771
Epoch 2/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7413 - loss: 0.4258 - val_AUC: 0.0000e+00 - val_loss: 1.0377
Epoch 3/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7580 - loss: 0.4176 - val_AUC: 0.0000e+00 - val_loss: 1.0435
Epoch 4/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - AUC: 0.7655 - loss: 0.4112 - val_AUC: 0.0000e+00 - val_loss: 0.9969
726/726 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Epoch 1/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - AUC: 0.6522 - loss: 0.4732 - val_AUC: 0.0000e+00 - val_loss: 1.0668
Epoch 2/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - AUC: 0.7405 - loss: 0.4291 - val_AUC: 0.0000e+00 - val_loss: 1.0162
Epoch 3/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7575 - loss: 0.4153 - val_AUC: 0.0000e+00 - val_loss: 0.9856
Epoch 4/50
654/654 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - AUC: 0.7630 - loss: 0.4111 - val_AUC: 0.0000e+00 - 

In [25]:
# Print results - ANN AUC
print("AUC ANN: {:.3f} %".format(cv_results_ann['test_auc'].mean() * 100))
print("AUC Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_auc'].std()))

AUC ANN: 75.617 %
AUC Standard Deviation ANN: 0.00512


In [26]:
# Print results - ANN Recall
print("Recall ANN: {:.4f}".format(cv_results_ann['test_recall'].mean()))
print("Recall Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_recall'].std()))

Recall ANN: 0.4478
Recall Standard Deviation ANN: 0.01253


In [27]:
# Print results - ANN Precision
print("Precision ANN: {:.4f}".format(cv_results_ann['test_precision'].mean()))
print("Precision Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_precision'].std()))

Precision ANN: 0.5689
Precision Standard Deviation ANN: 0.00659


In [28]:
# Print results - ANN F1 Score
print("F1-score ANN: {:.4f}".format(cv_results_ann['test_F1'].mean()))
print("F1-score Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_F1'].std()))

F1-score ANN: 0.6543
F1-score Standard Deviation ANN: 0.00385


#### 4.6.3 Cross Validation for the SVM Model

In [29]:
# Run cross-validation
cv_results_svm = cross_validate(
    estimator=pipeline,
    X=X_train_scaled,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [30]:
# Print results - SVM AUC
print("AUC SVM: {:.3f} %".format(cv_results_svm['test_auc'].mean() * 100))
print("AUC Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_auc'].std()))

AUC SVM: 76.256 %
AUC Standard Deviation SVM: 0.01477


In [31]:
# Print results - SVM Recall
print("Recall SVM: {:.4f}".format(cv_results_svm['test_recall'].mean()))
print("Recall Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_recall'].std()))

Recall SVM: 0.6828
Recall Standard Deviation SVM: 0.03114


In [32]:
# Print results - SVM Precision
print("Precision SVM: {:.4f}".format(cv_results_svm['test_precision'].mean()))
print("Precision Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_precision'].std()))

Precision SVM: 0.5128
Precision Standard Deviation SVM: 0.00775


In [33]:
# Print results - SVM F1 Score
print("F1-score SVM: {:.4f}".format(cv_results_svm['test_F1'].mean()))
print("F1-score Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_F1'].std()))

F1-score SVM: 0.6748
F1-score Standard Deviation SVM: 0.00950


#### 4.6.4 Cross Validation for the GBM Model

In [34]:
# Run cross-validation
cv_results_gbm = cross_validate(
    estimator=gbm,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [35]:
# Print results - GBM AUC
print("AUC GBM: {:.3f} %".format(cv_results_gbm['test_auc'].mean() * 100))
print("AUC Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_auc'].std()))

AUC GBM: 92.849 %
AUC Standard Deviation GBM: 0.10492


In [36]:
# Print results - GBM Recall
print("Recall GBM: {:.4f}".format(cv_results_gbm['test_recall'].mean()))
print("Recall Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_recall'].std()))

Recall GBM: 0.6960
Recall Standard Deviation GBM: 0.42127


In [37]:
# Print results - GBM Precision
print("Precision GBM: {:.4f}".format(cv_results_gbm['test_precision'].mean()))
print("Precision Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_precision'].std()))

Precision GBM: 0.8926
Precision Standard Deviation GBM: 0.14969


In [38]:
# Print results - GBM F1 Score
print("F1-score GBM: {:.4f}".format(cv_results_gbm['test_F1'].mean()))
print("F1-score Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_F1'].std()))

F1-score GBM: 0.8245
F1-score Standard Deviation GBM: 0.24062
